# Forward Spectrum Summary

Scientific question: which sampled wavelength has the minimum reflectance, and how closely does the Python-defined O2 A case track the committed DISAMAR reference spectrum?

This notebook is an executable demo. It builds the DISAMAR O2 A reference case from the repo helper, calls the Python API, writes disposable outputs under `out/demo/`, and leaves interpretation tables in the executed copy.

## Inputs and API Calls

The input is the Python-defined O2 A reference scene, including atmosphere, geometry, surface, aerosol, spectroscopy, CIA, and instrument-response controls. The API calls happen through a prepared O2 A context so diagnostics reuse the same resolved setup.

In [ ]:
from __future__ import annotations

import csv



def load_spectrum_csv(path: Path) -> dict[str, np.ndarray]:
    with path.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError(f"{path} is empty")
    return {
        key: np.array([float(row[key]) for row in rows], dtype=float)
        for key in rows[0].keys()
    }


def interpolate_to_grid(
    wavelength_nm: np.ndarray,
    reference: dict[str, np.ndarray],
) -> dict[str, np.ndarray]:
    reference_wavelength_nm = reference["wavelength_nm"]
    if (
        wavelength_nm[0] < reference_wavelength_nm[0]
        or wavelength_nm[-1] > reference_wavelength_nm[-1]
    ):
        raise ValueError("current spectrum grid extends outside the vendored DISAMAR reference grid")
    return {
        key: wavelength_nm if key == "wavelength_nm" else np.interp(wavelength_nm, reference_wavelength_nm, values)
        for key, values in reference.items()
    }


def write_spectrum_plot(
    output_path: Path,
    wavelength_nm: np.ndarray,
    radiance: np.ndarray,
    irradiance: np.ndarray,
    reflectance: np.ndarray,
    vendor: dict[str, np.ndarray],
    min_reflectance_index: int,
) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True, constrained_layout=True)
    fig.suptitle("DISAMAR O2 A parity route spectrum")

    axes[0].plot(wavelength_nm, vendor["reflectance"], label="Vendored DISAMAR reference", linewidth=1.8)
    axes[0].plot(wavelength_nm, reflectance, label="Python parity route", linewidth=1.4)
    axes[0].scatter(
        [wavelength_nm[min_reflectance_index]],
        [reflectance[min_reflectance_index]],
        color="#d62728",
        s=24,
        zorder=3,
        label="minimum reflectance",
    )
    axes[0].set_ylabel("reflectance")
    axes[0].legend(loc="best")

    axes[1].plot(wavelength_nm, vendor["radiance"], label="Vendored DISAMAR reference", linewidth=1.8)
    axes[1].plot(wavelength_nm, radiance, label="Python parity route", linewidth=1.4)
    axes[1].set_ylabel("radiance")
    axes[1].legend(loc="best")

    axes[2].plot(wavelength_nm, vendor["irradiance"], label="Vendored DISAMAR reference", linewidth=1.8)
    axes[2].plot(wavelength_nm, irradiance, label="Python parity route", linewidth=1.4)
    axes[2].set_ylabel("irradiance")
    axes[2].legend(loc="best")

    reflectance_residual = reflectance - vendor["reflectance"]
    axes[3].plot(wavelength_nm, reflectance_residual, color="tab:red", linewidth=1.3)
    axes[3].axhline(0.0, color="black", linewidth=0.8, alpha=0.7)
    axes[3].set_ylabel("reflectance\nresidual")
    axes[3].set_xlabel("wavelength (nm)")

    for axis in axes:
        axis.grid(True, alpha=0.25)
        axis.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3))

    fig.savefig(output_path, dpi=160)
    plt.close(fig)

from dataclasses import asdict
import json
import os
from pathlib import Path
import sys
import time

import numpy as np

from validation.common.o2a_reference_case import build_o2a_case

REPO_ROOT = Path(os.environ["ZDISAMAR_REPO_ROOT"]).resolve()
PYTHON_ROOT = REPO_ROOT / "python"
OUT_DIR = REPO_ROOT / "out" / "demo" / "forward_summary"
LIBRARY_NAME = "libzdisamar_c.dylib" if sys.platform == "darwin" else "libzdisamar_c.so"
LIBRARY_PATH = REPO_ROOT / "zig-out" / "lib" / LIBRARY_NAME
VENDOR_REFERENCE_PATH = REPO_ROOT / "validation" / "data" / "o2a_with_cia_disamar_reference.csv"
SUMMARY_OUTPUT = OUT_DIR / "python_forward_summary.json"
PLOT_OUTPUT = OUT_DIR / "python_forward_summary_plot.png"
TOLERANCE = 1.0e-12


def require_library() -> str:
    if not LIBRARY_PATH.exists():
        raise FileNotFoundError(f"{LIBRARY_PATH} does not exist; build the native shared library first")
    return str(LIBRARY_PATH)


def import_zdisamar():
    sys.path.insert(0, str(PYTHON_ROOT))
    import zdisamar as zd

    return zd


def run_forward_summary() -> dict:
    library_path = require_library()
    zd = import_zdisamar()
    case = build_o2a_case(zd)

    total_start = time.perf_counter()
    prepare_start = time.perf_counter()
    with zd.prepare(case, library_path=library_path) as prepared:
        prepare_s = time.perf_counter() - prepare_start
        forward_start = time.perf_counter()
        spectrum = prepared.forward_model()
        forward_s = time.perf_counter() - forward_start
        report = spectrum.diagnostic_report
        wavelength_nm = spectrum.wavelength_nm.copy()
        radiance = spectrum.radiance.copy()
        irradiance = spectrum.irradiance.copy()
        reflectance = spectrum.reflectance.copy()
        spectrum.close()

    min_index = int(np.argmin(reflectance))
    vendor = interpolate_to_grid(wavelength_nm, load_spectrum_csv(VENDOR_REFERENCE_PATH))
    reflectance_residual = reflectance - vendor["reflectance"]
    radiance_residual = radiance - vendor["radiance"]
    irradiance_residual = irradiance - vendor["irradiance"]

    write_spectrum_plot(
        PLOT_OUTPUT,
        wavelength_nm,
        radiance,
        irradiance,
        reflectance,
        vendor,
        min_index,
    )

    summary = {
        "sample_count": int(wavelength_nm.size),
        "min_reflectance": {
            "wavelength_nm": float(wavelength_nm[min_index]),
            "reflectance": float(reflectance[min_index]),
        },
        "diagnostic_report": asdict(report),
        "vendor_comparison": {
            "same_grid": bool(np.array_equal(wavelength_nm, vendor["wavelength_nm"])),
            "reflectance_mean_abs_residual": float(np.mean(np.abs(reflectance_residual))),
            "reflectance_max_abs_residual": float(np.max(np.abs(reflectance_residual))),
            "radiance_max_abs_residual": float(np.max(np.abs(radiance_residual))),
            "irradiance_max_abs_residual": float(np.max(np.abs(irradiance_residual))),
        },
        "timing": {
            "prepare_o2a_s": prepare_s,
            "forward_model_s": forward_s,
            "total_s": time.perf_counter() - total_start,
        },
        "artifacts": {
            "json": str(SUMMARY_OUTPUT),
            "plot": str(PLOT_OUTPUT),
        },
    }
    SUMMARY_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    SUMMARY_OUTPUT.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    return summary


def main() -> int:
    summary = run_forward_summary()
    timing = summary["timing"]
    residual = summary["vendor_comparison"]["reflectance_max_abs_residual"]
    min_reflectance = summary["min_reflectance"]
    print(
        f"plot={PLOT_OUTPUT} n={summary['sample_count']} "
        f"min={min_reflectance['reflectance']:.17g}@{min_reflectance['wavelength_nm']:.2f}nm "
        f"max_abs_residual={residual:.3e} "
        f"prepare={timing['prepare_o2a_s']:.2f}s forward={timing['forward_model_s']:.2f}s "
        f"total={timing['total_s']:.2f}s"
    )
    return 0 if residual <= TOLERANCE else 1



## Execute and Interpret

The cell below runs the demo, writes the table or plot artifacts, and returns the compact summary used for interpretation.

In [ ]:
summary = run_forward_summary()
summary
